[**Google Earth Engine**](https://earthengine.google.com/) is a public data archive of petabytes of historical satellite imagery and geospatial datasets. The advantage lies in its remarkable computation speed as processing is outsourced to Google servers. The platform provides a variety of constantly updated datasets; no download of raw imagery is required. While it is free of charge, one still needs to activate access to Google Earth Engine with a valid Google account.

The Jupyter Notebook uses the following [paper](https://www.mdpi.com/2071-1050/13/3/1042) to recreate the analysis for drought identification and trend analysis in Kenya using Google Earth Engine (GEE) for Python, follow these steps:

## Analysis Steps

### Step 1: Data Collection

Obtain long-term satellite-derived precipitation data using the CHIRPS dataset available in GEE. This data will be used to analyze drought conditions in Kenya.

### Step 2: Study Area Definition

Define the study area using Kenya's geographical boundaries.

### Step 3: Data Preprocessing

- **Extract Monthly Precipitation**: Extract monthly CHIRPS precipitation data for the study area to calculate Standardized Precipitation Index (SPI) at different time scales (e.g., SPI1, SPI3, SPI6, SPI12).
- **Clip to Region**: Clip the precipitation data to Kenya's boundaries to ensure that the analysis focuses solely on Kenya. This step has already been implemented using GEE.

### Step 4: Calculate Standardized Precipitation Index (SPI)

- **SPI Calculation**: Calculate SPI at different time scales (1-, 3-, 6-, and 12-month) for drought evaluation. Convert the CHIRPS precipitation data into SPI values by fitting a gamma distribution to each pixel's monthly time series.

### Step 5: Drought Characterization

- **Identify Drought Events**: Use run theory to identify drought events based on SPI values.
- **Drought Duration**: Identify the number of months with SPI values below thresholds like -1.0 for moderate drought.
- **Drought Severity**: Calculate the severity by summing all SPI values during the drought.
- **Drought Intensity**: Calculate drought intensity as severity divided by duration.

### Step 6: Trend Analysis

- **Mann-Kendall Test**: Implement the Mann-Kendall trend test to determine trends in SPI values at annual, seasonal, and monthly time scales.
- **Sen’s Slope Estimator**: Apply Sen’s slope estimator to understand the magnitude of the detected trends.

### Step 7: Clustering Analysis of Drought Metrics

This step aims to perform a comprehensive clustering analysis on drought metrics across administrative units in Kenya, including data preprocessing, dimensionality reduction, and clustering.

#### Workflow Steps

- **Data Preprocessing**:
  - Encode categorical features, and normalize numerical metrics using `MinMaxScaler`.
- **Feature Engineering**:
  - Construct a feature vector (`feature_vector`) for clustering, including encoded categorical features and  normalized numerical metrics.
- **Dimensionality Reduction Using PCA**:
  - Determine the optimal number of components for dimensionality reduction by analyzing cumulative explained variance and applying PCA accordingly.
- **Clustering Analysis**:
  - **K-Means Clustering**:
    - Determine the optimal number of clusters using the **Elbow Method** and apply K-Means.
    - Visualize the clusters geospatially.
  - **Hierarchical Clustering**:
    - Create a linkage matrix using **Ward's method** and determine the optimal clusters using a dendrogram.
    - Assign cluster labels and visualize geospatially.
  - **DBSCAN Clustering**:
    - Perform a grid search to determine the best parameters (`eps`, `min_samples`) and use the **Silhouette Score** for optimization.
    - Assign cluster labels and visualize geospatially.


### Step 8: EM-DAT Analysis

Compare historical drought years from the [Emergency Events Database (EM-DAT)](https://www.emdat.be/) to SPI-derived results.

### DISCLAIMER

This is a set of scripts  shared for educational purposes only.  Anyone who uses this code or its
functionality or structure, assumes full liability and credits the author.

#### Map Disclaimer

The designations employed and the presentation of the material on this map do not imply the expression
of any opinion whatsoever on the part of the author concerning the legal status of any country, territory, city or area or of its authorities, or concerning the delimitation of its
frontiers or boundaries.


In [1]:
#pip install pymannkendall
#!pip install ipympl

In [2]:
import ee
import geemap
import numpy as np
import pandas as pd
from scipy.stats import gamma, norm, kstest, probplot
import plotly.graph_objects as go
import time
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from ipywidgets import interact, Dropdown, fixed, widgets, Checkbox, RadioButtons
import pymannkendall as mk
import json
import geopandas as gpd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score

## TODO:  Define Analysis Boundaries
 

In [3]:
zambia_rain_df = pd.read_csv("zambia_admin2_rs_precip.csv")

print(zambia_rain_df.head(10))
print(zambia_rain_df.tail(10))


   Unnamed: 0  year  month                 date  region admin2_name  \
0           0  2000      1  2000-01-01T00:00:00  Zambia    Chibombo   
1           1  2000      2  2000-02-01T00:00:00  Zambia    Chibombo   
2           2  2000      3  2000-03-01T00:00:00  Zambia    Chibombo   
3          10  2000     11  2000-11-01T00:00:00  Zambia    Chibombo   
4          11  2000     12  2000-12-01T00:00:00  Zambia    Chibombo   
5          12  2001      1  2001-01-01T00:00:00  Zambia    Chibombo   
6          13  2001      2  2001-02-01T00:00:00  Zambia    Chibombo   
7          14  2001      3  2001-03-01T00:00:00  Zambia    Chibombo   
8          22  2001     11  2001-11-01T00:00:00  Zambia    Chibombo   
9          23  2001     12  2001-12-01T00:00:00  Zambia    Chibombo   

   precipitation  
0     216.899197  
1     242.517915  
2     195.530641  
3     117.328103  
4     201.903206  
5     222.092396  
6     331.400414  
7     160.703736  
8     134.069714  
9     199.873594  
      Unn

# Calculate Standardized Precipitation Index (SPI)

## SPI Definitions

* SPI1 (1-month scale)

Used to capture short-term monthly precipitation fluctuations and early warning of meteorological drought.

* SPI3 (3-month scale)

Useful for seasonal drought analysis, which helps in understanding the drought conditions over a quarterly period, closely related to agricultural impacts.

* SPI6 (6-month scale)

Provides insights into medium-term drought conditions, capturing both the end of one season and the start of another, which can influence soil moisture and crop yield over a longer period.

* SPI12 (12-month scale)

Used to assess long-term drought conditions, representing annual fluctuations and providing insights into the overall hydrological drought scenario, which can affect groundwater recharge and surface water availability.


## Selecting SPI Time Scales
Based on the above definitions we select the following time Scales for Long and Short Rains:

- Long Rains: March to June (4-month period) - We will use 1-month and 3-month SPI for long rains to understand short-term and medium-term droughts.
- Short Rains: October to December (3-month period) - We will use 1-month and 3-month SPI to capture variability within the shorter rainy season.

## Determining Gamma Distribution

What is a Gamma Distribution?

A gamma distribution is a type of statistical model used to describe data that are always positive and usually skewed, meaning most values are small but there can be a few large values. It's often used for things like rainfall, where:

- Many days have little or no rain, and
- A few days have a lot of rain.

This type of pattern creates a graph that has a high peak near the small values and a long tail extending to larger values. That’s what the gamma distribution looks like.

Why Do We Use Gamma Distribution for Rainfall?

- Always Positive: Rainfall cannot be negative; it either doesn’t rain or it rains some amount. The gamma distribution is great for representing only positive values.
- Right-Skewed Data: Most of the time, we have small amounts of rainfall, and occasionally, we get heavy rainfall. The gamma distribution is good for representing this kind of uneven distribution.
- Flexibility: The shape of the gamma distribution can change to fit different types of rainfall patterns, making it a very flexible tool for modeling different weather conditions.

The Gamma Distribution is the Standardized Precipitation Index (SPI) as a way to figure out if a place is having normal, dry, or wet weather compared to its history.

1. Modeling Historical Rainfall:

We use the gamma distribution to fit the historical rainfall data. This helps us understand what the usual rainfall is like for each month.

2. Comparing to Current Rainfall:

Once we have the gamma model of typical rainfall, we compare current rainfall to see how much it differs from the usual.
This comparison tells us if it is drier or wetter than normal, and by how much.

3. Getting SPI:

The final step is to convert this difference into a number called SPI, which tells us:
- If the SPI is negative, it means it’s drier than usual (possible drought).
- If the SPI is positive, it means it’s wetter than usual (possible flooding).

In [4]:
# Checking if all precipitation values are positive in each dataset
print("Zambia Rainy Season: All values positive? ", (zambia_rain_df['precipitation'] > 0).all())

Zambia Rainy Season: All values positive?  True


In [5]:
from utils.gamma_distribution_funcs import update_plot

# Get the list of unique Admin Level 2 regions (admin2_name)
admin2_names = zambia_rain_df['admin2_name'].unique()

# Create an interactive dropdown menu for selecting the admin2_name
dropdown = widgets.Dropdown(
    options=admin2_names,
    description='Select Region:',
    value=admin2_names[0]  # Default to the first region
)

# Create and display the interactive plot
interactive_plot = widgets.interactive(update_plot, admin2_name=dropdown, zambia_rain_df=widgets.fixed(zambia_rain_df))
display(interactive_plot)

interactive(children=(Dropdown(description='Select Region:', options=('Chibombo', 'Kabwe', 'Kapiri-Mposhi', 'M…

### Kolmogorov-Smirnov Test

Understanding the Kolmogorov-Smirnov Test:
The Kolmogorov-Smirnov test compares the empirical distribution of your data to a theoretical distribution (in this case, the gamma distribution).
The null hypothesis (H₀) of the KS test is that the data follows the specified distribution (in this case, gamma).
The alternative hypothesis (H₁) is that the data does not follow the specified distribution.

In the context of the Kolmogorov-Smirnov (KS) test, a p-value greater than 0.05 is typically desired when checking the fit of a distribution.

Interpreting the p-value:

- p-value > 0.05:
If the p-value is greater than 0.05, it means there is insufficient evidence to reject the null hypothesis.
In other words, the data fits the gamma distribution well. Therefore, we accept the null hypothesis and conclude that the gamma distribution is likely a good fit.

- p-value < 0.05:
If the p-value is less than 0.05, it means that there is significant evidence to reject the null hypothesis.
This suggests that the gamma distribution may not be a good fit for the data.

In [6]:
from utils.gamma_distribution_funcs import perform_ks_test


'''
def perform_ks_test(data, region_name):
    # Fit a gamma distribution to the data
    shape, loc, scale = gamma.fit(data, floc=0)  # floc=0 to ensure non-negative values

    # Perform Kolmogorov-Smirnov test
    test_stat, p_value = kstest(data, gamma(shape, loc, scale).cdf)

    # Print results for each region
    if p_value > 0.05:
        print(f"The data for {region_name} fits the gamma distribution well (p-value: {p_value:.4f}).")
    else:
        print(f"The gamma distribution may not be the best fit for {region_name} (p-value: {p_value:.4f}).")

'''


# Perform KS test for each admin1_name
for region_name, group in zambia_rain_df.groupby('admin2_name'):
    print(f"Testing for {region_name}:")
    perform_ks_test(group['precipitation'].dropna(), region_name)
    print()  # Add a line break between region results


Testing for Chadiza:
The data for Chadiza fits the gamma distribution well (p-value: 0.1303).

Testing for Chama:
The gamma distribution may not be the best fit for Chama (p-value: 0.0021).

Testing for Chavuma:
The data for Chavuma fits the gamma distribution well (p-value: 0.7088).

Testing for Chibombo:
The data for Chibombo fits the gamma distribution well (p-value: 0.3589).

Testing for Chienge:
The data for Chienge fits the gamma distribution well (p-value: 0.9080).

Testing for Chililabombwe:
The data for Chililabombwe fits the gamma distribution well (p-value: 0.1908).

Testing for Chilubi:
The data for Chilubi fits the gamma distribution well (p-value: 0.1109).

Testing for Chingola:
The data for Chingola fits the gamma distribution well (p-value: 0.0867).

Testing for Chinsali:
The gamma distribution may not be the best fit for Chinsali (p-value: 0.0027).

Testing for Chipata:
The data for Chipata fits the gamma distribution well (p-value: 0.0550).

Testing for Choma:
The dat

In [7]:
from utils.gamma_distribution_funcs import perform_ks_test_for_all


# Create a dropdown for the "pass_or_fail" option
pass_or_fail_dropdown = Dropdown(
    options=['pass', 'fail'],
    description='Pass or Fail:',
    style={'description_width': 'initial'}
)

# Interactive function to display the results of the KS test for all admin2 areas
interact(
    perform_ks_test_for_all,
    zambia_rain_df=widgets.fixed(zambia_rain_df),  # Pass zambia_rain_df as a fixed argument
    pass_or_fail=pass_or_fail_dropdown
)


interactive(children=(Dropdown(description='Pass or Fail:', options=('pass', 'fail'), style=DescriptionStyle(d…

<function utils.gamma_distribution_funcs.perform_ks_test_for_all(zambia_rain_df, pass_or_fail='pass')>

In [8]:
from utils.gamma_distribution_funcs import setup_interactive_widgets

# Ensure zambia_rain_df is loaded
setup_interactive_widgets(zambia_rain_df)


interactive(children=(Dropdown(description='Select Admin2:', options=('Chibombo', 'Kabwe', 'Kapiri-Mposhi', 'M…

## Calculate SPI

In [9]:
from utils.spi_funcs import calculate_spi_for_regions


# Calculate SPI for all regions
spi_results_df = calculate_spi_for_regions(zambia_rain_df)

# Display the results
print("SPI Results for all regions (admin2_name):")
print(spi_results_df.head())


SPI Results for all regions (admin2_name):
                     Chibombo_1_month  Chibombo_3_month  Kabwe_1_month  \
date                                                                     
2000-01-01T00:00:00          0.707354               NaN       0.712670   
2000-02-01T00:00:00          0.939230               NaN       0.870516   
2000-03-01T00:00:00          0.499873          1.107143       0.508235   
2000-11-01T00:00:00         -0.423085          0.505439      -0.185352   
2000-12-01T00:00:00          0.563240          0.239539       0.574574   

                     Kabwe_3_month  Kapiri_Mposhi_1_month  \
date                                                        
2000-01-01T00:00:00            NaN               0.648742   
2000-02-01T00:00:00            NaN               0.748236   
2000-03-01T00:00:00       1.077942               0.599916   
2000-11-01T00:00:00       0.582342              -0.465606   
2000-12-01T00:00:00       0.382142               0.956502   

          

In [10]:
from utils.spi_funcs import calculate_spi_for_regions, setup_interactive_plot_with_labels

setup_interactive_plot_with_labels(spi_results_df)

interactive(children=(Dropdown(description='Select Region & Scale:', options=('Chibombo_1_month', 'Chibombo_3_…

## Identify Drought Events

To characterize droughts, we need to analyze the SPI values calculated previously to identify distinct drought events. This involves analyzing when SPI values fall below a threshold, like -1.0, to determine the duration, severity, and intensity of each drought event.

Definitions:

- Drought Event: A period during which SPI is below a defined threshold (e.g., SPI < -1.0 for moderate drought).
- Drought Duration: The number of months during which the SPI value stays below the threshold.
- Drought Severity: The sum of all SPI values during the drought period.
- Drought Intensity: Severity divided by duration, indicating how intense the drought is on average.


**TODO**: 

In [11]:
from utils.drought_analysis_funcs import characterize_drought_events

drought_events_df = characterize_drought_events(spi_results_df)
drought_events_df

,admin2_name,time_scale,spi_scale,Drought Event,Drought Duration (months),Drought Severity,Drought Intensity,Drought Start,Drought End
0,Chibombo_1,month,1,1,9,-3.268724,-0.363192,2002-02-01T00:00:00,2002-11-01T00:00:00
1,Chibombo_1,month,1,2,1,-1.160000,-1.160000,2004-11-01T00:00:00,2004-12-01T00:00:00
2,Chibombo_1,month,1,3,9,-3.019505,-0.335501,2005-02-01T00:00:00,2005-11-01T00:00:00
3,Chibombo_1,month,1,4,8,-1.432502,-0.179063,2007-03-01T00:00:00,2007-11-01T00:00:00
4,Chibombo_1,month,1,5,0,-1.045599,0.000000,2009-02-01T00:00:00,2009-03-01T00:00:00
...,...,...,...,...,...,...,...,...,...
1901,Shang'ombo_3,month,3,4,1,-1.188577,-1.188577,2013-11-01T00:00:00,2013-12-01T00:00:00
1902,Shang'ombo_3,month,3,5,10,-5.190560,-0.519056,2015-03-01T00:00:00,2016-01-01T00:00:00
1903,Shang'ombo_3,month,3,6,12,-9.957015,-0.829751,2019-01-01T00:00:00,2020-01-01T00:00:00
1904,Shang'ombo_3,month,3,7,2,-3.406525,-1.703263,2021-11-01T00:00:00,2022-01-01T00:00:00


### Distribution Analysis

In [12]:
from utils.drought_analysis_funcs import setup_interactive_histogram

setup_interactive_histogram(drought_events_df)


interactive(children=(Dropdown(description='Admin2 Name:', options=('Chibombo_1', 'Chibombo_3', 'Kabwe_1', 'Ka…

### Drought Event Frequency

In [13]:
from utils.drought_analysis_funcs import setup_interactive_frequency_plot

# Call the setup function with your drought events DataFrame
setup_interactive_frequency_plot(drought_events_df)


interactive(children=(Dropdown(description='Select Dimension:', options=('admin2_name', 'spi_scale'), style=De…

### Adiministrative Level Comparison

In [14]:
from utils.drought_analysis_funcs import setup_interactive_drought_characteristics_plot

# Call the setup function
setup_interactive_drought_characteristics_plot(drought_events_df)


interactive(children=(Dropdown(description='Select Characteristic:', options=('Drought Intensity', 'Drought Se…

### Correlation Analysis

In [15]:
from utils.drought_analysis_funcs import setup_interactive_correlation_analysis_plot


setup_interactive_correlation_analysis_plot(drought_events_df)


interactive(children=(Dropdown(description='Select X Variable:', options=('Drought Duration (months)', 'Drough…

### SPI Scale (1-month vs. 3-month) Analysis

In [18]:
from utils.drought_analysis_funcs import setup_interactive_drought_analysis_custom


setup_interactive_drought_analysis_custom(drought_events_df)


interactive(children=(Dropdown(description='Select Characteristic:', options=('Drought Duration (months)', 'Dr…

### Temproal Heatmap